# 03. Build the Smallest Useful CNN Baseline

Do not begin with a transformer or diffusion model because a paper title sounds impressive. A simple baseline tells you whether the data pipeline, target, and metric contain a learnable signal.

In [ ]:
import torch
from torch import nn

class SmallRestorationCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 3, padding=1),
        )
    def forward(self, x):
        return self.net(x)

model = SmallRestorationCNN()
print(sum(p.numel() for p in model.parameters()), 'parameters')

## Tiny-data overfit test

Before a long run, ask whether the model can memorize 4–16 examples. Failure to overfit a tiny set is often a pipeline/loss/optimization bug, not evidence that you need a larger model.

In [ ]:
torch.manual_seed(0)
target = torch.rand(4, 1, 32, 32)
input_ = torch.clamp(target + 0.1 * torch.randn_like(target), 0, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.L1Loss()

for step in range(40):
    optimizer.zero_grad(set_to_none=True)
    pred = model(input_)
    loss = loss_fn(pred, target)
    loss.backward()
    optimizer.step()
print('final tiny-set loss:', float(loss))

If the tiny-set loss does not fall, inspect shapes, normalization, alignment, activation/output range, learning rate, and the target before increasing complexity.